> 在线讲义：[第 2 章 · 行为克隆：照着人类示范抄作业](https://wikieden.github.io/vla-from-scratch/lessons/02-behavior-cloning.html) · 本书《VLA 从零到 π0》

# 第 2 章 · 行为克隆：照着人类示范抄作业

配套讲义：仓库里 `vla-deep-dive/notes/lessons/02-behavior-cloning.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `l1_loss(pred, target)` | 7 维里只有一维差 0.01：L1 = 0.0014 |
| 第 2 步 | `rollout(coverage)` | 覆盖 ±0.1：最后偏 4.33；覆盖 ±0.5：回到 0.00 |

复合误差用一维小车模拟：状态是偏离车道中线多少；示范只覆盖了 |偏离| ≤ 覆盖宽度 的状态。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

**函数 `l1_loss`**：行为克隆的 L1 损失：模型动作和人的动作差多少（ACT 用 L1）。

In [ ]:
def l1_loss(pred, target):
    """行为克隆的 L1 损失：模型动作和人的动作差多少（ACT 用 L1）。"""
    return float(np.abs(np.asarray(pred, float) - np.asarray(target, float)).mean())

**函数 `rollout`**：一条轨迹：见过的状态里会往回拉；没见过的状态只能乱猜，偏差越滚越大。

In [ ]:
def rollout(coverage, steps=40, bump_at=12, bump=0.3, seed=0):
    """一条轨迹：见过的状态里会往回拉；没见过的状态只能乱猜，偏差越滚越大。"""
    rng = np.random.default_rng(seed)
    x, path = 0.0, []
    for t in range(steps):
        if t == bump_at:
            x += bump  # 被碰歪一下
        if abs(x) <= coverage:
            act = -0.5 * x  # 训练数据里见过：照着示范往中线拉回
        else:
            act = np.sign(x) * 0.15 + rng.normal(0, 0.05)  # 没见过：乱猜，还往外偏
        x += act
        path.append(x)
    return np.array(path)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 看图答题：L1 损失")
print(f"  模型 [0, 0, -0.02, 0, 0, 0, 0] vs 人 [0, 0, -0.01, 0, 0, 0, 0]：L1 = "
      f"{l1_loss([0, 0, -0.02, 0, 0, 0, 0], [0, 0, -0.01, 0, 0, 0, 0]):.4f}")

In [ ]:
print("\n第二步 复合误差：被碰歪后见没见过")
for name, cov in (("纯 BC（只录正中间）", 0.1), ("加了偏了再拉回的示范", 0.5)):
    p = rollout(cov)
    print(f"  {name}：最大偏离 {np.abs(p).max():.2f}，最后偏离 {abs(p[-1]):.2f}")

## 3. 代码里才看得到的

1. `l1_loss` 对 7 维取平均，所以一维差 0.01 只算 0.01 / 7。
2. `rollout` 的"没见过就乱猜"用固定往外偏 0.15 + 噪声模拟，是示意，不是真实策略。

## 4. 自己改一改

- 把 `bump` 从 0.3 改成 0.05，看纯 BC 还会不会冲出去。
- 把覆盖宽度从 0.1 慢慢加到 0.5，找出刚好能拉回的那个值。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 人做一遍记一遍，看图答题对答案；
> 只学人走的路，一走偏就是没见过的路，越猜越偏。